# Brunel University London

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import pandas as pd

# Start the Selenium Chrome driver
driver = webdriver.Chrome()

# Base URL of the page
base_url = 'https://www.brunel.ac.uk/study/Course-listing?courseLevel=&courseSearch=&page={}'

# Initialize an empty list to store all course data
all_course_data = []

# Loop through pages
page_number = 1
while True:
    driver.get(base_url.format(page_number))

    # Handle cookie consent if it appears
    try:
        cookie_consent = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.ID, 'CybotCookiebotDialog'))
        )
        driver.execute_script("arguments[0].style.display='none';", cookie_consent)
    except:
        pass

    # Find the table element
    table = driver.find_element(By.ID, 'responsive-example-table')

    # Find all rows in the table
    rows = table.find_elements(By.TAG_NAME, 'tr')

    # Loop through each row to extract data
    for row in rows[1:]:
        try:
            columns = row.find_elements(By.TAG_NAME, 'td')

            # Extracting data from each column in the row, handling exceptions
            try:
                course_name = columns[0].find_element(By.TAG_NAME, 'a').text
            except NoSuchElementException:
                course_name = 'NA'

            try:
                course_url = columns[0].find_element(By.TAG_NAME, 'a').get_attribute('href')
            except NoSuchElementException:
                course_url = 'NA'

            try:
                ucas_pg_code = columns[1].text
            except NoSuchElementException:
                ucas_pg_code = 'NA'

            try:
                study_mode = columns[2].text
            except NoSuchElementException:
                study_mode = 'NA'

            try:
                level = columns[3].text
            except NoSuchElementException:
                level = 'NA'

            # Appending the extracted data to the list as a dictionary
            all_course_data.append({
                'Course_Title': course_name,
                'Degree_Level': level,
                'Attendance_Mode': study_mode,
                'Course_Link': course_url,
            })
        except Exception as e:
            print(f"An error occurred: {e}")

    # Move to the next page or break the loop if there is no 'next' button
    try:
        next_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//li[@class="arrow"]/a[@title="Go to the next page of the results"]'))
        )
        next_button.click()
        page_number += 1
    except NoSuchElementException:
        break
    except ElementClickInterceptedException:
        driver.execute_script("arguments[0].click();", next_button)
        page_number += 1
    except Exception as e:
        break

# Create a DataFrame from all the extracted data
df = pd.DataFrame(all_course_data)

# Display the DataFrame
df

# Close the browser
driver.quit()


In [6]:
df

,Course_Title,Degree_Level,Attendance_Mode,Course_Link
0,Accountancy BSc,Undergraduate,3 years full-time; 4 years full-time,https://www.brunel.ac.uk/study/undergraduate/a...
1,Accounting and Business Intelligence MSc (Brun...,Postgraduate,1 year full-time; 2 years full-time ; 16 month...,https://www.brunel.ac.uk/study/postgraduate/ac...
2,Accounting and Business Management BSc (Brunel...,Undergraduate,3 years full-time; 4 years full-time,https://www.brunel.ac.uk/study/undergraduate/a...
3,Accounting and Business Management MSc (Brunel...,Postgraduate,1 year full-time; 2 years full-time ; 16 month...,https://www.brunel.ac.uk/study/postgraduate/ac...
4,Advanced Clinical Practice (Cardiovascular Hea...,Postgraduate,3 years part-time; 3 years part-time; 1 year (...,https://www.brunel.ac.uk/study/postgraduate/ad...
...,...,...,...,...
330,Undergraduate Pathway in Life Sciences,Foundation,2 semesters full-time,https://www.brunel.ac.uk/study/undergraduate/u...
331,Visual Effects and Motion Graphics BSc,Undergraduate,3 years full-time; 4 years full-time,https://www.brunel.ac.uk/study/undergraduate/v...
332,Water and Environmental Engineering MSc,Postgraduate,1 year full-time; 14 months full-time,https://www.brunel.ac.uk/study/postgraduate/wa...
333,"Welfare, Health and Wellbeing PhD",PhD & Research,3 years full-time; 6 years part-time,https://www.brunel.ac.uk/study/postgraduate/we...


In [9]:
df.to_csv('BrunelUniversityLondon.csv',index=False)